<a href="https://colab.research.google.com/github/ubiai-incorporated/specialized_language_models_book/blob/master/chapter3_complete_finetuning_methods.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Fine-Tuning Methods :Complete Implementation



## Setup: Install All Required Libraries

In [ ]:
# Install all required packages
!pip install torch transformers datasets peft bitsandbytes accelerate evaluate trl -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 17.9 MB/s eta 0:00:00


---
# Method 1: Selective Layer Freezing



In [ ]:
import torch
from torch.optim import AdamW
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from torch.utils.data import DataLoader, TensorDataset
from datasets import load_dataset
import numpy as np

model_name = "bert-base-uncased"
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"Loaded model: {model_name}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loaded model: bert-base-uncased
Total parameters: 109,483,778


In [ ]:
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer
import torch

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def create_dummy_dataset(num_samples=100):
    texts  = [f"This is sample text {i}" for i in range(num_samples)]
    labels = [i % 2 for i in range(num_samples)]

    # Tokenize here — output is already tensors, not strings
    encodings = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    dataset = TensorDataset(
        encodings["input_ids"],
        encodings["attention_mask"],
        torch.tensor(labels)
    )
    return dataset

train_dataset    = create_dummy_dataset(100)
val_dataset      = create_dummy_dataset(20)

train_dataloader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_dataloader   = DataLoader(val_dataset,   batch_size=8)

batch = next(iter(train_dataloader))
input_ids, attention_mask, labels = batch


In [ ]:
dataset = load_dataset("rotten_tomatoes")

# Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["text"])
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format("torch")

# Create dataloaders
train_dataloader = DataLoader(tokenized_dataset["train"], batch_size=8, shuffle=True)
val_dataloader = DataLoader(tokenized_dataset["validation"], batch_size=8)

print(f"Train samples: {len(tokenized_dataset['train'])}")
print(f"Validation samples: {len(tokenized_dataset['validation'])}")

Loading Rotten Tomatoes dataset...


Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Train samples: 8530
Validation samples: 1066


In [ ]:
# Listing 3.4: Implementing layer freezing technique
num_layers = len(model.bert.encoder.layer)
bottom_to_freeze = 6  # Freeze bottom 6 layers

print(f"Total layers: {num_layers}")
print(f"Freezing bottom {bottom_to_freeze} layers\n")

for i, layer in enumerate(model.bert.encoder.layer):
    if i < bottom_to_freeze:
        for param in layer.parameters():
            param.requires_grad = False
        print(f"Layer {i}: FROZEN")
    else:
        print(f"Layer {i}: TRAINABLE")

Total layers: 12
Freezing bottom 6 layers

Layer 0: FROZEN
Layer 1: FROZEN
Layer 2: FROZEN
Layer 3: FROZEN
Layer 4: FROZEN
Layer 5: FROZEN
Layer 6: TRAINABLE
Layer 7: TRAINABLE
Layer 8: TRAINABLE
Layer 9: TRAINABLE
Layer 10: TRAINABLE
Layer 11: TRAINABLE


In [ ]:
# Listing 3.5: Implement partial layer freezing technique
# Partial freezing in upper layers (example: freeze specific attention heads)
# Note: This is conceptual - BERT doesn't expose individual heads as separate parameters
# In practice, you'd freeze entire attention layers or sublayers
print("\nPartial freezing in upper layers:")
for i, layer in enumerate(model.bert.encoder.layer[bottom_to_freeze:], start=bottom_to_freeze):
    # Freeze attention output dense layer as an example of partial freezing
    for param in layer.attention.output.dense.parameters():
        param.requires_grad = False
    print(f"Layer {i}: Partially frozen (attention output)")


Partial freezing in upper layers:
Layer 6: Partially frozen (attention output)
Layer 7: Partially frozen (attention output)
Layer 8: Partially frozen (attention output)
Layer 9: Partially frozen (attention output)
Layer 10: Partially frozen (attention output)
Layer 11: Partially frozen (attention output)


In [ ]:
# Listing 3.6: Define the parameters to train
# Only include trainable parameters in optimizer
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW(trainable_params, lr=2e-5)

print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in trainable_params):,}")
print(f"Trainable %: {100 * sum(p.numel() for p in trainable_params) / sum(p.numel() for p in model.parameters()):.2f}%")


Total parameters: 109,483,778
Trainable parameters: 63,412,994
Trainable %: 57.92%


In [ ]:
# Listing 3.7: Training loop
num_epochs = 5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"\nTraining on device: {device}")
print(f"Starting training for {num_epochs} epochs...\n")

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for batch_idx, batch in enumerate(train_dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()

    avg_loss = total_loss / len(train_dataloader)
    print(f"Epoch {epoch + 1}/{num_epochs} - Average Loss: {avg_loss:.4f}")

    # Listing 3.8: Unfreezing top layers after third iteration
    # Progressive unfreezing: unfreeze top layers after epoch 3
    if epoch == 2:  # After 3rd epoch (0-indexed)
        print("\n=== UNFREEZING TOP LAYERS ===")
        for i, layer in enumerate(model.bert.encoder.layer[bottom_to_freeze:], start=bottom_to_freeze):
            for param in layer.parameters():
                param.requires_grad = True
            print(f"Layer {i}: UNFROZEN")

        # Recreate optimizer with newly unfrozen parameters
        trainable_params = [p for p in model.parameters() if p.requires_grad]
        optimizer = AdamW(trainable_params, lr=2e-5)
        print(f"Updated trainable parameters: {sum(p.numel() for p in trainable_params):,}\n")

print("\nTraining complete!")

In [ ]:
# Listing 3.9: Check the freezing status at the end
print("\n=== Final Trainable Status ===")
for i, layer in enumerate(model.bert.encoder.layer):
    trainable = any(p.requires_grad for p in layer.parameters())
    print(f"Layer {i}: {'TRAINABLE' if trainable else 'FROZEN'}")


=== Final Trainable Status ===
Layer 0: FROZEN
Layer 1: FROZEN
Layer 2: FROZEN
Layer 3: FROZEN
Layer 4: FROZEN
Layer 5: FROZEN
Layer 6: TRAINABLE
Layer 7: TRAINABLE
Layer 8: TRAINABLE
Layer 9: TRAINABLE
Layer 10: TRAINABLE
Layer 11: TRAINABLE


In [ ]:
# Evaluate the model
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch in val_dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        predictions = torch.argmax(outputs.logits, dim=-1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total
print(f"\nValidation Accuracy: {accuracy:.4f} ({100*accuracy:.2f}%)")

---
# Method 2: LoRA (Low-Rank Adaptation)



LoRA introduces low-rank matrices for parameter-efficient fine-tuning with <1% trainable parameters.

In [ ]:
# Listing 3.10: Importing required libraries
import time
from datasets import load_dataset
from peft import PeftModel, PeftConfig, LoraConfig, TaskType, get_peft_model
from transformers import (
    TrainingArguments,
    Trainer,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding
)
import evaluate
import numpy as np

print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
# Listing 3.11: Mapping labels with numerical ids and viceversa
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

print("Label mappings:")
print(f"id2label: {id2label}")
print(f"label2id: {label2id}")

Label mappings:
id2label: {0: 'NEGATIVE', 1: 'POSITIVE'}
label2id: {'NEGATIVE': 0, 'POSITIVE': 1}


In [ ]:
# Listing 3.12: Load the pre-trained model
checkpoint = "distilbert-base-uncased"
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

print(f"Model loaded: {checkpoint}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded: distilbert-base-uncased
Parameters: 66,955,010


In [ ]:
# Listing 3.13: Loading the dataset
dataset = load_dataset("rotten_tomatoes")

print(f"Train samples: {len(dataset['train'])}")
print(f"Validation samples: {len(dataset['validation'])}")
print(f"Test samples: {len(dataset['test'])}")
print(f"\nSample: {dataset['train'][0]}")

Train samples: 8530
Validation samples: 1066
Test samples: 1066

Sample: {'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .', 'label': 1}


In [ ]:
# Listing 3.14: Text tokenization
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenizer_func(input):
    return tokenizer(input["text"], truncation=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
data_tokenized = dataset.map(tokenizer_func, batched=True)

print("Dataset tokenized successfully!")
print(f"Tokenized sample keys: {data_tokenized['train'][0].keys()}")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Dataset tokenized successfully!
Tokenized sample keys: dict_keys(['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'])


In [ ]:
# Listing 3.15: Renaming columns and removing those unnecessary
train_data_tokenized = data_tokenized["train"].remove_columns(["text"]).rename_column("label", "labels")
val_data_tokenized = data_tokenized["validation"].remove_columns(["text"]).rename_column("label", "labels")

print(f"Train dataset columns: {train_data_tokenized.column_names}")
print(f"Validation dataset columns: {val_data_tokenized.column_names}")

Train dataset columns: ['labels', 'input_ids', 'token_type_ids', 'attention_mask']
Validation dataset columns: ['labels', 'input_ids', 'token_type_ids', 'attention_mask']


In [ ]:
# Listing 3.16: Configuring LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_lin", "v_lin"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

print(f"\nLoRA Configuration:")
print(f"  Rank (r): {lora_config.r}")
print(f"  Alpha: {lora_config.lora_alpha}")
print(f"  Target modules: {lora_config.target_modules}")
print(f"  Dropout: {lora_config.lora_dropout}")

trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925

LoRA Configuration:
  Rank (r): 8
  Alpha: 32
  Target modules: {'v_lin', 'q_lin'}
  Dropout: 0.05


In [ ]:
# Listing 3.17: Configuring the training arguments
output_dir = f'./rotten-tomatoes-classification-training-{str(int(time.time()))}'
training_args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=50,
)

print(f"Training arguments configured:")
print(f"  Output directory: {output_dir}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Epochs: {training_args.num_train_epochs}")

Training arguments configured:
  Output directory: ./rotten-tomatoes-classification-training-1774892937
  Learning rate: 0.0001
  Batch size: 16
  Epochs: 3


In [ ]:
# Define metrics
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_data_tokenized,
    eval_dataset=val_data_tokenized,
    processing_class=tokenizer,   # ← replaces 'tokenizer'
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("\n=== Starting LoRA Training ===")
trainer.train()

# Evaluate
results = trainer.evaluate()
print(f"\nLoRA Results:")
print(f"  Accuracy: {results['eval_accuracy']:.4f}")
print(f"  Loss: {results['eval_loss']:.4f}")

# Save adapter
peft_model.save_pretrained("./lora_adapter")
print("\nLoRA adapter saved to ./lora_adapter")

---
# Method 3: QLoRA (Quantized LoRA)

**From Chapter 3 - Listings 3.19 to 3.31**

QLoRA combines 4-bit quantization with LoRA for extreme memory efficiency.

In [ ]:
# Listing 3.19: Importing required libraries
import time
import torch
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    TrainingArguments,
    Trainer,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    BitsAndBytesConfig
)
import numpy as np
import evaluate

print("Libraries imported for QLoRA!")

Libraries imported for QLoRA!


In [ ]:
# Listing 3.20: Load Rotten Tomatoes dataset and mapping the labels
data = load_dataset("rotten_tomatoes")
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

print(f"Dataset loaded:")
print(f"  Train: {len(data['train'])} samples")
print(f"  Validation: {len(data['validation'])} samples")

Dataset loaded:
  Train: 8530 samples
  Validation: 1066 samples


In [ ]:
# Listing 3.21: Configuring the double quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("4-bit quantization config:")
print(f"  Double quantization: {bnb_config.bnb_4bit_use_double_quant}")
print(f"  Quantization type: {bnb_config.bnb_4bit_quant_type}")
print(f"  Compute dtype: {bnb_config.bnb_4bit_compute_dtype}")

4-bit quantization config:
  Double quantization: True
  Quantization type: nf4
  Compute dtype: torch.bfloat16


In [ ]:
# Listing 3.22: Load the model and apply the quantization configuration
checkpoint = "bert-base-uncased"
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    quantization_config=bnb_config,
    device_map="auto"
)

print(f"\nQuantized model loaded: {checkpoint}")
print(f"Model device: {model.device}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Quantized model loaded: bert-base-uncased
Model device: cpu


In [ ]:
# Listing 3.23: Preparing the model for k-bit training and enabling gradient checkpoint
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()

print("Model prepared for k-bit training")
print("Gradient checkpointing enabled")

Model prepared for k-bit training
Gradient checkpointing enabled


In [ ]:
# Listing 3.24: Loading the tokenizer
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenizer_func(input):
    return tokenizer(input["text"], truncation=True, max_length=512)

print("Tokenizer loaded")

Tokenizer loaded


In [ ]:
# Listing 3.25: Define data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
data_tokenized = data.map(tokenizer_func, batched=True)

print("Dataset tokenized")

Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Dataset tokenized


In [ ]:
# Listing 3.26: Split dataset in training and validation
train_data_tokenized = data_tokenized["train"].remove_columns(["text"]).rename_column("label", "labels")
val_data_tokenized = data_tokenized["validation"].remove_columns(["text"]).rename_column("label", "labels")

print(f"Training samples: {len(train_data_tokenized)}")
print(f"Validation samples: {len(val_data_tokenized)}")

Training samples: 8530
Validation samples: 1066


In [ ]:
# Listing 3.27: Defining LoRA configuration
lora_config = LoraConfig(
    r=16,  # rank
    lora_alpha=32,  # scaling factor
    target_modules=["query", "value"],  # BERT uses different naming than DistilBERT
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_CLS,
    modules_to_save=["classifier"]  # Keep classifier layer trainable
)

print("QLoRA configuration:")
print(f"  Rank: {lora_config.r}")
print(f"  Alpha: {lora_config.lora_alpha}")
print(f"  Target modules: {lora_config.target_modules}")

QLoRA configuration:
  Rank: 16
  Alpha: 32
  Target modules: {'query', 'value'}


In [ ]:
# Listing 3.28: Creating the peft model from the original
peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

print("\nQLoRA model created successfully")

trainable params: 591,362 || all params: 110,075,140 || trainable%: 0.5372

QLoRA model created successfully


In [ ]:
# Listing 3.29: Creating the training arguments
import torch

output_dir = f'./rotten-tomatoes-qlora-training-{str(int(time.time()))}'

# Detect hardware capabilities
cuda_available = torch.cuda.is_available()
bf16_supported = cuda_available and torch.cuda.is_bf16_supported()

training_args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=2e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=cuda_available and not bf16_supported,  # fp16 if GPU but no bf16
    bf16=bf16_supported,                         # bf16 only if Ampere+ GPU
    gradient_checkpointing=True,
    optim="paged_adamw_8bit" if cuda_available else "adamw_torch",  # paged_adamw_8bit requires CUDA
    report_to="none"
)

print(f"Training arguments:")
print(f"  Output:    {output_dir}")
print(f"  Device:    {'CUDA - ' + torch.cuda.get_device_name(0) if cuda_available else 'CPU'}")
print(f"  Optimizer: {training_args.optim}")
print(f"  bf16:      {training_args.bf16}")
print(f"  fp16:      {training_args.fp16}")

Training arguments:
  Output:    ./rotten-tomatoes-qlora-training-1774893299
  Device:    CPU
  Optimizer: OptimizerNames.ADAMW_TORCH
  bf16:      False
  fp16:      False


In [ ]:
# Define metrics
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
# Listing 3.30: Creating the Trainer
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_data_tokenized,
    eval_dataset=val_data_tokenized,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print("Trainer created")

Trainer created


In [ ]:
# Listing 3.31: Train
print("\n=== Starting QLoRA Training ===")
trainer.train()

# Evaluate
results = trainer.evaluate()
print(f"\nQLoRA Results:")
print(f"  Accuracy: {results['eval_accuracy']:.4f}")
print(f"  Loss: {results['eval_loss']:.4f}")

# Save adapter
peft_model.save_pretrained("./qlora_adapter")
print("\nQLoRA adapter saved to ./qlora_adapter")

---
# Method 4: DoRA (Weight-Decomposed LoRA)



DoRA decomposes weights into magnitude and direction for more focused adaptation.

In [ ]:
# Listing 3.32: Configuring DoRA
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments, DataCollatorWithPadding
from datasets import load_dataset
from peft import TaskType
import time
import numpy as np
import evaluate

# Load dataset
dataset = load_dataset("rotten_tomatoes")
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

# Load model and tokenizer
checkpoint = "distilbert-base-uncased"
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

print(f"Model loaded: {checkpoint}")
print(f"Dataset loaded: {len(dataset['train'])} training samples")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded: distilbert-base-uncased
Dataset loaded: 8530 training samples


In [ ]:
# Initialize DoRA configuration
config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_lin", "v_lin"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_CLS,
    use_dora=True  # Enable DoRA
)

# Apply DoRA to model
dora_model = get_peft_model(model, config)
dora_model.print_trainable_parameters()

print("\nDoRA Configuration:")
print(f"  DoRA enabled: {config.use_dora}")
print(f"  Rank: {config.r}")
print(f"  Alpha: {config.lora_alpha}")

trainable params: 748,802 || all params: 67,703,812 || trainable%: 1.1060

DoRA Configuration:
  DoRA enabled: True
  Rank: 8
  Alpha: 32


In [ ]:
# Tokenize dataset
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True)

tokenized_dataset = dataset.map(tokenize_function, batched=True)
train_dataset = tokenized_dataset["train"].remove_columns(["text"]).rename_column("label", "labels")
val_dataset = tokenized_dataset["validation"].remove_columns(["text"]).rename_column("label", "labels")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Dataset tokenized")

Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Dataset tokenized


In [ ]:
# Training arguments
output_dir = f'./dora-training-{str(int(time.time()))}'
training_args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=50,
)

# Define metrics
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# Create trainer
trainer = Trainer(
    model=dora_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer configured for DoRA")

Trainer configured for DoRA


In [ ]:
# Train DoRA model
print("\n=== Starting DoRA Training ===")
trainer.train()

# Evaluate
results = trainer.evaluate()
print(f"\nDoRA Results:")
print(f"  Accuracy: {results['eval_accuracy']:.4f}")
print(f"  Loss: {results['eval_loss']:.4f}")

# Save adapter
dora_model.save_pretrained("./dora_adapter")
print("\nDoRA adapter saved to ./dora_adapter")

---
# Method 5: NEFTune (Noisy Embeddings Fine-Tuning)


NEFTune adds noise to embeddings during training to improve instruction following.

In [ ]:
# Listing 3.33: Configuring the trainer to use NEFT
from trl import SFTTrainer, SFTConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType
import torch
import time

In [ ]:

# ── 1. Device & precision ─────────────────────────────────────────────────────
cuda_available = torch.cuda.is_available()
bf16_supported = cuda_available and torch.cuda.is_bf16_supported()
device = torch.device("cuda" if cuda_available else "cpu")

In [ ]:

# ── 2. Model & tokenizer ──────────────────────────────────────────────────────
model_name = "gpt2"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id

print(f"Model loaded: {model_name}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded: gpt2
Parameters: 124,439,808


In [ ]:

# ── 3. Dataset ────────────────────────────────────────────────────────────────
dataset = load_dataset("rotten_tomatoes")

def format_instruction(example):
    sentiment = "positive" if example["label"] == 1 else "negative"
    return {"text": f"Review: {example['text']}\nSentiment: {sentiment}"}

formatted_dataset = dataset.map(format_instruction, remove_columns=["label"])

print(f"\nDataset formatted for instruction tuning")
print(f"Sample:\n{formatted_dataset['train'][0]['text']}")

Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]


Dataset formatted for instruction tuning
Sample:
Review: the rock is destined to be the 21st century's new " conan " and that he's going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .
Sentiment: positive


In [ ]:
# ── 4. LoRA ───────────────────────────────────────────────────────────────────
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["c_attn"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("\nLoRA applied to GPT-2")

trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.2364

LoRA applied to GPT-2


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [ ]:
# ── 5. Tokenize dataset ───────────────────────────────────────────────────────
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=128,
        padding="max_length"
    )

tokenized_train = formatted_dataset["train"].map(tokenize, batched=True)
tokenized_val   = formatted_dataset["validation"].map(tokenize, batched=True)

# ── 6. SFTConfig (replaces TrainingArguments + SFTTrainer kwargs) ─────────────
output_dir = f'./neftune-training-{str(int(time.time()))}'

Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

In [ ]:
training_args = SFTConfig(
    output_dir=output_dir,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    bf16=bf16_supported,
    fp16=cuda_available and not bf16_supported,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit" if cuda_available else "adamw_torch",
    neftune_noise_alpha=5,
    max_length=128,
    report_to="none"
)

print(f"Training arguments configured")
print(f"NEFTune noise alpha: {training_args.neftune_noise_alpha}")


Training arguments configured
NEFTune noise alpha: 5


In [ ]:
# ── 7. Trainer ────────────────────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    args=training_args,        # SFTConfig carries all kwargs — no extras needed
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
)

Truncating train dataset:   0%|          | 0/8530 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1066 [00:00<?, ? examples/s]

In [ ]:
# ── 8. Train ──────────────────────────────────────────────────────────────────
print("\n=== Starting NEFTune Training ===")
print("Noise is added to embeddings during training only")
print("This improves instruction-following ability\n")

trainer.train()

results = trainer.evaluate()
print(f"\nNEFTune Results:")
print(f"  Loss: {results['eval_loss']:.4f}")

In [ ]:
# ── 9. Save ───────────────────────────────────────────────────────────────────
model.save_pretrained("./neftune_model")
print("\nNEFTune model saved to ./neftune_model")

# ── 10. Inference test ────────────────────────────────────────────────────────
model.to(device)
model.eval()

test_prompt = "Review: This movie was absolutely fantastic! The acting was superb.\nSentiment:"
inputs = tokenizer(test_prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=10,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"\nTest Generation:")
print(f"Input:  {test_prompt}")
print(f"Output: {generated_text}")